# Imports

In [ ]:
# Jupyter Notebook Utils
%matplotlib inline
%load_ext autoreload
%autoreload 2

# Python Libraries
import pandas as pd
import numpy as np
import matplotlib.dates as mdates
from datetime import date
import matplotlib.pyplot as plt
import seaborn as sns
import math
from statsmodels.stats import diagnostic
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Own modules
from src import *

# Data Sanity Check

In [ ]:
check_quality(interactions_df)
check_quality(embeddings_subreddits_df)
check_quality(embeddings_users_df)

In [ ]:
interactions_df

# Patient-0 Analysis

## Data Focusing tool

In order to identify the patient 0, we first construct a tool capable of focus on a time window and on links which contain a specific keyword. This allows us to dig into a more relevant part of the data, and therefore identify more easily trends in a specific context. 

We highlight that this method is meant to help faster identification of trends, and that a more rigorous causal analysis will be conducted later on. This is important as this focusing tool may focus too much on parts of the data, and thus eliminate precursors without eliminating their effect. In instance, the actual patient zero subreddit or community might not contain the chosen given in its name, leaving the possibility of having it as a precursor behind. 

In [ ]:
# Subreddit keyword to focus on
KEYWORD = "nba"

# Time Window to consider
# TIME_WINDOW = [date(2016, 6, 1), date(2017, 2, 1)] # TRUMP
TIME_WINDOW = [date(2015, 6, 1), date(2015, 8, 1)] # NBA

# Extract focused dadaframe
focused_df = extract_focused_df(interactions_df, KEYWORD, TIME_WINDOW)
focused_df.head(10)

In this example we focus on the NBA keyword and over the course of two months from June 2015 to August 2015. For further information about how the extract_focused_df() function works, please refer to the associated .py file. Note that the focused_df preserves the order of the given dataframe in entry, that is time in our case.

## Analysis of specific subreddits interactions over a given time period
### Number of daily interactions across each subreddit
We start by conducting a temporal analysis of all interactions, as source or as target. This allows us to identify interaction outbreaks, which will be further analyzed to see if they are positive, negative and/or toxic events.

In [ ]:
print("There are", len(focused_df["SOURCE_SUBREDDIT"].unique()), "different subreddits acting as source.")
print("There are", len(focused_df["TARGET_SUBREDDIT"].unique()), "different subreddits acting as target.")

# Count the number of daily interactions across all considered subreddits
daily_interaction_df = construct_daily_interaction_df(focused_df, KEYWORD, TIME_WINDOW)

# Plot
fig, ax = plt.subplots(figsize=(10,4))

daily_interaction_df[["DAILY_SOURCE_INTERACTIONS",
                      "DAILY_TARGET_INTERACTIONS",
                      "DAILY_TOTAL_INTERACTIONS"]].plot(ax=ax)

# ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
ax.tick_params(axis="x", rotation=45)
ax.set_title("Interactions over time")
ax.set_xlabel("Day")
ax.set_ylabel("Number of Interactions")
ax.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

We can see that a clear outbreak appears at the beginning of July 2015. We can also see that overall there are more targeted subreddits than source (a smaller amount of subreddits target more other subreddits). But during the outbreak, the number of source subreddits explodes, indicating a lot more different subreddits linked to the same (small) amount of target subreddits. This could be a toxic wave coming from different communities towards one particular.

### Number of interactions (as source and target) for each subreddit 

To further analyze which subreddits are potential victims and prosecutors, we show the amount of interactions (as source and as target) for the 8 most interactive subreddits.

In [ ]:
# Count the number of interaction as source and as target for each subreddit
sr_interaction_count_df = construct_interaction_count_df(focused_df)

# Choose the number of subreddits to plot
NB_TO_PLOT = 8

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

sr_interaction_count_df.head(NB_TO_PLOT).plot(
    x="SUBREDDIT",
    y=["SOURCE_INTERACTIONS", "TARGET_INTERACTIONS", "TOTAL_INTERACTIONS"],
    kind="bar",
    width=0.8,
    ax=ax
)

ax.set_title("Top " + str(NB_TO_PLOT) + " Subreddits by Number of Interactions", fontsize=14)
ax.set_xlabel("Subreddit", fontsize=12)
ax.set_ylabel("Number of Interactions", fontsize=12)
ax.set_xticklabels(sr_interaction_count_df.head(NB_TO_PLOT)["SUBREDDIT"], rotation=45, ha="right")
ax.legend(["As Source", "As Target", "Total"], title="Interaction Type")
ax.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

A clear trend of links towards the 'nba' subreddit shows up, and some others like the 'nbacirclejerk' appears as one of the most targeting subreddit. Although, the balance between subreddits is not equitable and therefore if any toxic outbreak were to be happening, this would mean many attackers are implied rather than a specific subreddit, which makes the analysis more complicated. We have to analyze the evolution interactions for each small subreddit separately to see if it is indeed a change in their behaviour or a recurrent interaction between the 'victim' and the 'prosecutor'. 

### Number of negative daily interactions across each subreddit 

We now study the interactions themselves to see if they are negative and/or toxic based on as many criteria as wanted. Here, we take:
- 'LINK_SENTIMENT' == -1,
- 'COMPOUND_SENTIMENT_CALCULATED_BY_VADER' <= -0.5,
- 'NEGATIVE_SENTIMENT_CALCULATED_BY_VADER' >= 0.1

These are very non-strict thresholds, for the sake of the example.  

In [ ]:
# Count the number of negative daily interactions across all considered subreddits
CRITERION = ["LINK_SENTIMENT", 
             "COMPOUND_SENTIMENT_CALCULATED_BY_VADER",
             "NEGATIVE_SENTIMENT_CALCULATED_BY_VADER"
             ]
THRESHOLDS = [-0.5,
              -0.5,
              0.10
              ]
UPPER = [True,
         True,
         False
         ]
daily_neg_interaction_df = construct_neg_interaction_df(focused_df, 
                                                        time_window=TIME_WINDOW, 
                                                        criteria=CRITERION, 
                                                        threshold=THRESHOLDS, 
                                                        upper=UPPER,
                                                        groupby_type="DAY")

# Plot
fig, ax = plt.subplots(figsize=(10,4))

daily_neg_interaction_df.plot(ax=ax)

ax.xaxis.set_major_locator(mdates.DayLocator(interval=7))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
ax.tick_params(axis="x", rotation=45)
ax.set_title("Negative Interactions over time")
ax.set_xlabel("Day")
ax.set_ylabel("Number of Negative Interactions")
ax.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# Show dataframe for further investigation
daily_neg_interaction_df

It appears that during the outbreak, 28 of the 35 interactions have a negative sentiment calculated by VADER above 0.1, but only 1 interaction has a negative link_sentiment. In this example, we would tend to say this is not a major negative outbreak, but rather an outbreak with regular amount of toxicity. Note again that this should and will be further analyzed to quantify the regular amount of toxicity and compare it with the outbreak, in the causal analysis.

### Number of interactions (as source and target) for each subreddit

Finally, we add a plot showing the number of negative interactions for each subreddit which allow us to highlight which subreddit is associated with negative links.

In [ ]:
# Count the number of negative interactions for each considered subreddit

# Compute negative interactions as source
sr_neg_source_interaction_df = construct_neg_interaction_df(focused_df, 
                                                        time_window=TIME_WINDOW, 
                                                        criteria=CRITERION, 
                                                        threshold=THRESHOLDS, 
                                                        upper=UPPER,
                                                        groupby_type="SOURCE_SUBREDDIT")
# Compute negative interactions as target
sr_neg_target_interaction_df = construct_neg_interaction_df(focused_df, 
                                                        time_window=TIME_WINDOW, 
                                                        criteria=CRITERION, 
                                                        threshold=THRESHOLDS, 
                                                        upper=UPPER,
                                                        groupby_type="TARGET_SUBREDDIT")
# Add interactions and sort them by total
sr_neg_interaction_df = smart_df_add(sr_neg_source_interaction_df, sr_neg_target_interaction_df)
sr_neg_interaction_df["TOTAL_NEGATIVE_INTERACTIONS"] = sr_neg_interaction_df.sum(axis=1)
sr_neg_interaction_df = sr_neg_interaction_df.sort_values(by="TOTAL_NEGATIVE_INTERACTIONS", ascending=False)

# Choose the number of subreddits to plot
NB_TO_PLOT = 9

# Plot
fig, ax = plt.subplots(figsize=(8, 5))

sr_neg_interaction_df.head(NB_TO_PLOT).plot(
    kind="bar",
    width=0.8,
    ax=ax
)

ax.set_title("Top " + str(NB_TO_PLOT) + " Subreddits by Number of Interactions", fontsize=14)
ax.set_xlabel("Subreddit", fontsize=12)
ax.set_ylabel("Number of Interactions", fontsize=12)
ax.set_xticklabels(sr_interaction_count_df.head(NB_TO_PLOT)["SUBREDDIT"], rotation=45, ha="right")
ax.legend(sr_neg_interaction_df.columns, title="Interaction Type")
ax.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

Again, the 'nba' subreddit stands out which makes sense as it was the most interactive during this period. We can see that the total amount of negative interactions goes up to roughly 130 over 250 total interactions, which should be shown in parallel with the habitual toxic interaction ratio, and with the relevance of each criterion. Also, this total negative interactions should not be trusted as it most certainly count very toxic interactions several times (flagged as toxic by every criterion). We cannot therefore assess if this was a negative outbreak for the 'nba' subreddit.

## Causal Analysis

This allows us to perform a more trustful study based on the prior beliefs obtained in the previous section. This includes assessing relevance of criteria and thresholds and global toxic trends of each subreddit in this context among others.